# Model Evaluation Notebook
This notebook loads the dataset, applies the same preprocessing and downsampling strategy used during training, and evaluates the saved `.pt` models on the Test Set. It plots the training loss and F1-score over time, and generates an evaluation report.

In [ ]:
import sys
sys.path.append('.')

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

from data_preparation import DataPreparation
from loader.dataset_factory import DatasetFactory
from model.gagnn import GAGNN
from model.loss import GAGNNLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from utils import Evaluator

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. Data Loading and Preparation

In [ ]:
date_time_1 = '2022-09-07 14:55:00'
date_time_2 = '2022-09-08 16:12:00'

print("Loading dataset...")
loader = DatasetFactory.get_loader(
    "ibm_amlsim", 
    engine="pyarrow", 
    dtype_backend="pyarrow", 
    nrows=None
)
loader.load()
transactions_df = loader.get_transactions()

# Se fosse un problema multiclasse, useresti:
# data_prep = DataPreparation(scaler_type='robust')
# transactions_df = data_prep.parse_patterns_file(transactions_df, 'data/ibm_amlsim/HI-Small_Patterns.txt')

ts_raw = pd.to_datetime(transactions_df['Timestamp'])
thresh_val = pd.to_datetime(date_time_1)
train_mask_raw = ts_raw < thresh_val

data_prep = DataPreparation(scaler_type='robust')
edges_features_df = data_prep.fit_transform_edges(transactions_df, train_mask=train_mask_raw)
nodes_features_df = data_prep.get_node_features(edges_features_df, train_mask=train_mask_raw)
print("Data loaded and prepared.")

### 2. Downsampling & PyTorch Geometric Data Setup

In [ ]:
print("--- Downsampling Strategy (1:1 Ratio) ---")
fraud_nodes = nodes_features_df[nodes_features_df["Is Laundering"] > 0].index
legit_nodes = nodes_features_df[nodes_features_df["Is Laundering"] == 0].index

sampled_legit_nodes = pd.Series(legit_nodes).sample(n=len(fraud_nodes), random_state=42).values
sampled_users = set(fraud_nodes).union(set(sampled_legit_nodes))

nodes_features_df = nodes_features_df.loc[list(sampled_users)]
edges_features_df = edges_features_df[
    edges_features_df['Account'].isin(sampled_users) & 
    edges_features_df['Account.1'].isin(sampled_users)
]

unique_nodes = nodes_features_df.index.unique()
node_mapping = pd.Series(index=unique_nodes, data=np.arange(len(unique_nodes)))

src = edges_features_df['Account'].map(node_mapping).values
dst = edges_features_df['Account.1'].map(node_mapping).values
edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

edge_features_cols = [c for c in edges_features_df.columns if c not in ['Account', 'Account.1', 'Is Laundering', 'Timestamp']]
edge_attr = torch.tensor(edges_features_df[edge_features_cols].values, dtype=torch.float)
y_trans = torch.tensor(edges_features_df['Is Laundering'].values, dtype=torch.float).unsqueeze(1)

node_features_cols = [c for c in nodes_features_df.columns if c != 'Is Laundering']
x = torch.tensor(nodes_features_df[node_features_cols].values, dtype=torch.float)
y_node = torch.tensor(nodes_features_df['Is Laundering'].values, dtype=torch.float)

ts = pd.to_datetime(edges_features_df['Timestamp'])
thresh_val = pd.to_datetime(date_time_1)
thresh_test = pd.to_datetime(date_time_2)

train_edge_mask = torch.tensor((ts < thresh_val).values, dtype=torch.bool)
val_edge_mask = torch.tensor(((ts >= thresh_val) & (ts < thresh_test)).values, dtype=torch.bool)
test_edge_mask = torch.tensor((ts >= thresh_test).values, dtype=torch.bool)

train_nodes_idx = edge_index[:, train_edge_mask].flatten().unique()
train_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
train_node_mask[train_nodes_idx] = True

val_nodes_idx = edge_index[:, val_edge_mask].flatten().unique()
val_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
val_node_mask[val_nodes_idx] = True

test_nodes_idx = edge_index[:, test_edge_mask].flatten().unique()
test_node_mask = torch.zeros(x.shape[0], dtype=torch.bool)
test_node_mask[test_nodes_idx] = True

node_mask = torch.ones(x.shape[0], dtype=torch.bool)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y_trans=y_trans, y_node=y_node)
data.node_mask = node_mask
data.edge_train_mask = train_edge_mask
data.edge_val_mask = val_edge_mask
data.edge_test_mask = test_edge_mask
data.train_node_mask = train_node_mask
data.val_node_mask = val_node_mask
data.test_node_mask = test_node_mask

batch_size = 512
test_loader = NeighborLoader(
    data, 
    num_neighbors=[-1, -1], 
    batch_size=batch_size,
    input_nodes=data.test_node_mask, 
    shuffle=False
)
print(f"Test loader initialized. Total Test batches: {len(test_loader)}")

### 3. Evaluation Function

In [ ]:
def evaluate_model(model_path, params_path, task='binary'):
    print("=" * 60)
    print(f"Evaluating Model: {model_path}")
    print("=" * 60)
    
    if not os.path.exists(model_path) or not os.path.exists(params_path):
        print(f"Files not found! Make sure '{model_path}' and '{params_path}' exist.")
        return
    
    # Load best hyperparameters to recreate the Loss function
    with open(params_path, "r") as f:
        best_params = json.load(f)
        
    # Load model and plot training history
    eval_model = GAGNN.load_saved(model_path).to(device)
    eval_model.plot_training_history()
    
    eval_criterion = GAGNNLoss(
        c1=best_params.get('c1', 1.0), 
        c2=best_params.get('c2', 1.0), 
        c3=best_params.get('c3', 1.0),
        laundry_weight=best_params.get('laundry_weight', 1.0)
    )
    
    # Evaluate
    Evaluator.evaluation_report(
        eval_model, 
        test_loader, 
        eval_criterion, 
        device, 
        edge_mask_name='edge_test_mask',
        task=task
    )


### 4. Evaluate Models

In [ ]:
# Esempio di utilizzo (modifica i percorsi con quelli dei tuoi modelli salvati):
# evaluate_model('Binary/saved_models_GAT/model_trained_final.pt', 'Binary/saved_models_GAT/best_params.json', task='binary')
# evaluate_model('Multiclass/saved_models_GAT/model_trained_final.pt', 'Multiclass/saved_models_GAT/best_params.json', task='multiclass')
